In [11]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import time

In [12]:
# Récupération du contenu de la page d'acceuil du site de vente
url="https://www.europe-camions.com/tracteur-occasion/1-31/annonces-tracteur.html"
contenu_site = requests.get(url)
extract_contenu = BeautifulSoup(contenu_site.text, 'html.parser') 
"""
On définit un dictionnnaire vide ayant comme clé les caractérique de la publication que nous cherchons à scraper.
"""
data = {"Description": [], "Kilométrage": [], "Date de 1ere immatriculation": [], "Energie": [],
        "Marque":[], "Modèle":[], "Pays":[], "Prix":[], "Vendeur":[], "Localisation":[], "Photo":[],
        "Poids à vide":[], "PTC":[], "Essieux":[], "Puissance":[]}

# Première page
"""_summary_
Après inspection de la page, nous remarquons que les blocs d'informations que nous cherchons à scraper sont tous regroupés dans des div de classe "google-tag". 
Nous allons donc parcourir tous ces blocs et extraire les informations qui nous intéressent.
"""
info_vehicule = extract_contenu.find_all('div', class_='google-tag') 
for info_pub in info_vehicule:
    
    prix = info_pub.find("span", class_="meta-price")
    if prix:
        data["Prix"].append(prix.get_text(strip=True))
    
    vendeur = info_pub.find("span", class_ ="title-vendeur")
    if vendeur:
        data["Vendeur"].append(vendeur.get_text(strip=True))
    
    
    geo = info_pub.find("span", class_="d-none")
    if geo:
        data["Localisation"].append(geo.get_text(strip=True).replace("-", ""))
    
    
    titre = info_pub.select_one("h2")
    if titre:
        data["Description"].append(titre.get_text(strip=True))
    
    img = info_pub.select_one("img")
    if img and img.has_attr("src"):
        data["Photo"].append(img.get("src"))

    """
    Lorsqu'on déroule les éléments de page, on voit que les info sont contenues dans des balises td et que toutes les caractéristiques sont composées par deux td succesives
    td 1 : caractéristique
    td 2 : valeur
    Nous ce que intéresse c'est le td 2, pour cela on parcours toutes td et on met une condition sur le td à l'instant t.
    Si la valeur du td est dans la liste des clées de notre dictionnaires alors forcément le td suivant qui est la valeurs fait partie des caractéristiques à scarper.
    """
    description = info_pub.find_all("td")
    for index, value in enumerate(description):
        # si la balise td courante contient une des caractéristiques que nous cherchons à scraper alors on ajoute la valeur du td suivant à notre dictionnaire
        if value.get_text(strip=True) in data.keys():
            value_suivante = description[index + 1]
            if value_suivante:
                data[value.get_text(strip=True)].append(value_suivante.get_text(strip=True))
            continue

In [13]:
pages_vues = set("https://www.europe-camions.com/tracteur-occasion/1-31/annonces-tracteur.html")
while url:
    if url in pages_vues:  # si on a déjà visité cette page, on arrête
        print(f"Page déjà vue : {url}, fin du scraping")
        break
    pages_vues.add(url)  # ajouter la page actuelle à l'ensemble des pages vues
    contenu_site = requests.get(url)
    
    extract_contenu = BeautifulSoup(contenu_site.text, 'html.parser') 

    info_vehicule = extract_contenu.find_all('div', class_='google-tag') 
    for info_pub in info_vehicule:
        
        prix = info_pub.find("span", class_="meta-price")
        if prix:
            data["Prix"].append(prix.get_text(strip=True))
        
        vendeur = info_pub.find("span", class_ ="title-vendeur")
        if vendeur:
            data["Vendeur"].append(vendeur.get_text(strip=True))
        
        
        geo = info_pub.find("span", class_="d-none")
        if geo:
            data["Localisation"].append(geo.get_text(strip=True).replace("-", ""))
        
        
        titre = info_pub.select_one("h2")
        if titre:
            data["Description"].append(titre.get_text(strip=True))
        
        img = info_pub.select_one("img")
        if img and img.has_attr("src"):
            data["Photo"].append(img.get("src"))

        
        description = info_pub.find_all("td")
        for index, value in enumerate(description):
            # si la balise td courante contient une des caractéristiques que nous cherchons à scraper alors on ajoute la valeur du td suivant à notre dictionnaire
            if value.get_text(strip=True) in data.keys():
                value_suivante = description[index + 1]
                if value_suivante:
                    data[value.get_text(strip=True)].append(value_suivante.get_text(strip=True))
                continue
            

   
    #Pause pour ne pas surcharger le serveur
    time.sleep(0.5)
    # Pagination : chercher le bouton "next"
    next_button = extract_contenu.select_one("div.btn-next-pagination.next.btn.text-center.no-decoration")
    if next_button and next_button.has_attr("data-ihref"):
        next_page = next_button["data-ihref"]
        print(f"Page suivante trouvée : {next_page}")
        url = "https://www.europe-camions.com" + next_page
    else:
        url = None

Page suivante trouvée : /tracteur-occasion/1-31/annonces-tracteur.html?p=2
Page suivante trouvée : /tracteur-occasion/1-31/annonces-tracteur.html?p=3
Page suivante trouvée : /tracteur-occasion/1-31/annonces-tracteur.html?p=4
Page suivante trouvée : /tracteur-occasion/1-31/annonces-tracteur.html?p=5
Page suivante trouvée : /tracteur-occasion/1-31/annonces-tracteur.html?p=6
Page suivante trouvée : /tracteur-occasion/1-31/annonces-tracteur.html?p=7
Page suivante trouvée : /tracteur-occasion/1-31/annonces-tracteur.html?p=8
Page suivante trouvée : /tracteur-occasion/1-31/annonces-tracteur.html?p=9
Page suivante trouvée : /tracteur-occasion/1-31/annonces-tracteur.html?p=10
Page suivante trouvée : /tracteur-occasion/1-31/annonces-tracteur.html?p=11
Page suivante trouvée : /tracteur-occasion/1-31/annonces-tracteur.html?p=12
Page suivante trouvée : /tracteur-occasion/1-31/annonces-tracteur.html?p=13
Page suivante trouvée : /tracteur-occasion/1-31/annonces-tracteur.html?p=14
Page suivante trouvé

In [14]:
df = pd.DataFrame(data.values(), index=data.keys()).T
df["Puissance"] = df["Puissance"].str.extract(r'(\d+)')
df.shape

(10033, 15)

In [15]:
df.head(30)

,Description,Kilométrage,Date de 1ere immatriculation,Energie,Marque,Modèle,Pays,Prix,Vendeur,Localisation,Photo,Poids à vide,PTC,Essieux,Puissance
0,Tracteur convoi exceptionnel Iveco Stralis 560,869 277 km,15/05/2008,Gazoil,Iveco,560,FRANCE,10 500 EURHT,VENIMAT,Rhône,https://photo.static-viamobilis.com/31/1/11151...,8.53 Tonnes,19 Tonnes,4x2,560
1,Tracteur DAF XF 530,559 816 km,12/06/2019,Gazoil,DAF,530,FRANCE,33 290 EURHT,AUTO PIECE BURGUIERE,Aveyron,https://photo.static-viamobilis.com/31/1/95449...,7.32 Tonnes,19 Tonnes,4x2,530
2,Tracteur MAN TGX 18.470 BL SA +ADR+INTARDER,408 453 km,26/07/2021,Gazoil,MAN,460,FRANCE,36 850 EURHT,BRAEM FRANCE SA,Nord,https://photo.static-viamobilis.com/31/1/11176...,7.4 Tonnes,19 Tonnes,4x2,470
3,Tracteur Mercedes Actros Arocs 1846 LS (no ),572 129 km,05/04/2017,Gazoil,Mercedes,I SAVE 500,FRANCE,17 850 EURHT,BRAEM FRANCE SA,Nord,https://photo.static-viamobilis.com/31/1/11176...,7500 Tonnes,19 Tonnes,4x2,460
4,Tracteur MAN TGX 18.520,68 271 km,17/09/2024,Gazoil,MAN,1848 L,FRANCE,79 000 EURHT,AUTO EXPORT IVANOV,BouchesduRhône,https://photo.static-viamobilis.com/31/1/11176...,7900 Tonnes,38 Tonnes,4x2,480
5,Tracteur Renault T-Series,355 000 km,01/07/2022,Gazoil,Renault,410,FRANCE,69 000 EURHT,BERNIS TRUCKS POITIERS,Vienne,https://photo.static-viamobilis.com/31/1/11176...,8 Tonnes,19 Tonnes,4x2,460
6,Tracteur Volvo FH13 460,196 191 km,02/08/2023,Gazoil,Volvo,1851 L,FRANCE,Prix sur demande,VOLVO TRUCK CENTER ILE DE FRANCE MASSY,Essonne,https://photo.static-viamobilis.com/31/1/11010...,8.53 Tonnes,19 Tonnes,4x2,500
7,Tracteur Volvo FH13 I SAVE 500,414 259 km,01/02/2023,Gazoil,Volvo,730,FRANCE,Prix sur demande,VOLVO TRUCK CENTER ILE DE FRANCE MASSY,Essonne,https://photo.static-viamobilis.com/31/1/11110...,7.32 Tonnes,60 Tonnes,4x2,480
8,Tracteur Mercedes Actros 1848 L,125 000 km,02/12/2022,Gazoil,Mercedes,450,FRANCE,68 000 EURHT,Mercedes-Benz Trucks BYmyCAR Côte d’Azur,Var,https://photo.static-viamobilis.com/31/1/11152...,7.4 Tonnes,18 Tonnes,4x2,410
9,Tracteur Scania G 410,161 607 km,08/11/2016,Gazoil,Scania,480 FT,FRANCE,44 000 EURHT,ST PREMIUM,Gironde,https://photo.static-viamobilis.com/31/1/11001...,7500 Tonnes,18 Tonnes,4x2,510
